# Mozilla Common Voice 15.0 Arabic: Exploratory Data Analysis
**Repository:** `fsicoli/common_voice_15_0` · **Configuration:** `ar` · **Primary split:** `train`

This Colab notebook performs full-split metadata and transcript analysis, then decodes a configurable representative audio sample (500 clips by default). It covers Arabic text normalization, vocabulary, contributor and demographic metadata, community votes, audio characteristics, text-audio relationships, and data-quality audits.

The repository describes an unofficial converted version of Common Voice 15.0. Its documented fields include audio, sentence, accent, age, client_id, up_votes, down_votes, gender, locale, and segment. The notebook inspects the actual schema and conditionally skips unavailable fields.

**Coverage distinction:** text and metadata analysis cover the complete training split; audio signal analysis covers the selected sample. Contributor IDs are treated as dataset identifiers, not real-world identities.

## 0. Configuration
Adjust these values before running. Optional validation/test loading is disabled by default to limit storage and download time.

In [ ]:
DATASET_ID = "fsicoli/common_voice_15_0"
CONFIG_NAME = "ar"
PRIMARY_SPLIT = "train"
LOAD_VALIDATION_AND_TEST = False
OPTIONAL_SPLITS = ["validation", "test"]

RANDOM_SEED = 42
AUDIO_SAMPLE_SIZE = 500
AUDIO_SAMPLING_STRATEGY = "random"  # "random" or "stratified"
STRATIFY_COLUMN = "client_id"

SAVE_TO_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CommonVoice15_Arabic_EDA"
LOCAL_OUTPUT_DIR = "/content/common_voice15_arabic_eda_outputs"

TOP_N_WORDS = 30
TOP_N_CHARACTERS = 35
N_AUDIO_EXAMPLES = 3
MAX_WAVEFORM_SECONDS = 20
FIGURE_DPI = 160

## 1. Install and import dependencies
The dataset card documents a loading-script-based dataset. This pinned Datasets version is included for legacy script compatibility. Review and trust the repository code before enabling remote dataset code.

In [ ]:
%pip -q install "datasets[audio]==2.19.2" librosa soundfile seaborn scipy reportlab nbconvert

In [ ]:
import re, json, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Audio, display, Markdown
from datasets import load_dataset
import librosa
import librosa.display
import soundfile as sf
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2. Output folders and Google Drive

In [ ]:
if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
    except Exception as exc:
        print("Drive mount failed; using local output:", exc)
        OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)
else:
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)

PLOTS_DIR, TABLES_DIR, REPORT_DIR = OUTPUT_DIR/"plots", OUTPUT_DIR/"tables", OUTPUT_DIR/"reports"
for folder in [OUTPUT_DIR, PLOTS_DIR, TABLES_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)

## 3. Download the complete Arabic training split
The dataset card uses `ar` as the Arabic configuration. Audio is excluded from the metadata dataframe below to avoid decoding every recording during transcript analysis.

In [ ]:
train_ds = load_dataset(
    DATASET_ID, CONFIG_NAME, split=PRIMARY_SPLIT,
    trust_remote_code=True
)
print(train_ds)
print("Rows:", len(train_ds))
print("Features:")
for name, feature in train_ds.features.items():
    print(name, ":", feature)

## 4. Optional validation and test splits
Enable the configuration flag if you want to compare split sizes and transcript statistics. Missing splits are reported without stopping the notebook.

In [ ]:
additional_splits = {}
if LOAD_VALIDATION_AND_TEST:
    for split_name in OPTIONAL_SPLITS:
        try:
            additional_splits[split_name] = load_dataset(
                DATASET_ID, CONFIG_NAME, split=split_name,
                trust_remote_code=True
            )
            print(split_name, len(additional_splits[split_name]))
        except Exception as exc:
            print(f"Could not load {split_name}: {exc}")
else:
    print("Optional split loading is disabled.")

## 5. Metadata table and schema overview

In [ ]:
AUDIO_COL = "audio" if "audio" in train_ds.column_names else None
metadata_ds = train_ds.remove_columns(AUDIO_COL) if AUDIO_COL else train_ds
df = metadata_ds.to_pandas()

overview = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percent": [float(df[c].isna().mean()*100) for c in df.columns],
    "unique_non_null": [int(df[c].nunique(dropna=True)) for c in df.columns]
})
display(overview)
overview.to_csv(TABLES_DIR/"column_overview.csv", index=False)
print("Rows:", len(df), "| columns:", len(df.columns))
print("Duplicate complete metadata rows:", df.duplicated().sum())

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(overview.column.astype(str), overview.missing_percent)
ax.set_title("Missing values by metadata column")
ax.set_ylabel("Missing (%)")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"missing_values.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()

## 6. Detect transcript and metadata fields
Common Voice generally uses `sentence` for text and `client_id` for the contributor identifier. The notebook detects these fields and prints the actual mapping.

In [ ]:
def first_existing(candidates, columns):
    return next((x for x in candidates if x in columns), None)

TRANSCRIPT_COL = first_existing(["sentence", "transcription", "raw_transcription", "text"], df.columns)
SPEAKER_COL = first_existing(["client_id", "speaker_id", "speaker"], df.columns)
GENDER_COL = first_existing(["gender", "sex"], df.columns)
AGE_COL = first_existing(["age"], df.columns)
ACCENT_COL = first_existing(["accent"], df.columns)
LOCALE_COL = first_existing(["locale"], df.columns)
UPVOTES_COL = first_existing(["up_votes", "upvotes"], df.columns)
DOWNVOTES_COL = first_existing(["down_votes", "downvotes"], df.columns)

detected = {
    "transcript": TRANSCRIPT_COL, "client/speaker": SPEAKER_COL,
    "gender": GENDER_COL, "age": AGE_COL, "accent": ACCENT_COL,
    "locale": LOCALE_COL, "up_votes": UPVOTES_COL, "down_votes": DOWNVOTES_COL
}
for key, value in detected.items():
    print(f"{key:18s}: {value}")
if TRANSCRIPT_COL is None:
    raise ValueError("Transcript field not found. Inspect df.columns and set TRANSCRIPT_COL.")

## 7. Original and lightly normalized Arabic transcripts
Original sentences are preserved. A separate analysis version removes tatweel/diacritics, normalizes Alef forms and Alef Maqsura, and cleans whitespace. This is an exploratory normalization, not an official WER reference normalization.

In [ ]:
DIACRITICS_RE = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")

def normalize_arabic_light(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("ـ", "")
    text = DIACRITICS_RE.sub("", text)
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    return re.sub(r"\s+", " ", text).strip()

df["transcript_original"] = df[TRANSCRIPT_COL].fillna("").astype(str)
df["transcript_normalized"] = df.transcript_original.map(normalize_arabic_light)
df["tokens"] = df.transcript_normalized.map(lambda x: x.split() if x else [])
df["word_count"] = df.tokens.map(len)
df["character_count"] = df.transcript_original.map(len)
df["normalized_character_count"] = df.transcript_normalized.map(len)
df["unique_word_count"] = df.tokens.map(lambda x: len(set(x)))
display(df[["transcript_original", "transcript_normalized", "word_count"]].head(12))

## 8. Transcript length statistics

In [ ]:
text_cols = ["word_count", "character_count", "normalized_character_count", "unique_word_count"]
text_summary = df[text_cols].describe(percentiles=[.05, .25, .5, .75, .95]).T
display(text_summary)
text_summary.to_csv(TABLES_DIR/"transcript_length_summary.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, title in zip(axes, ["word_count","character_count","unique_word_count"],
                          ["Words per sentence","Characters per sentence","Unique words per sentence"]):
    sns.histplot(df[col], bins=30, kde=True, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(PLOTS_DIR/"transcript_length_distributions.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()

## 9. Vocabulary, frequent tokens, bigrams, and trigrams
Whitespace tokenization is transparent but does not segment Arabic clitics or morphology.

In [ ]:
all_tokens = [token for tokens in df.tokens for token in tokens]
word_counts = Counter(all_tokens)
vocab_summary = pd.DataFrame([
    ("Total whitespace tokens", len(all_tokens)),
    ("Vocabulary size", len(word_counts)),
    ("Type-token ratio", len(word_counts)/len(all_tokens) if all_tokens else np.nan),
    ("Mean words per sentence", df.word_count.mean())
], columns=["metric","value"])
display(vocab_summary)
vocab_summary.to_csv(TABLES_DIR/"vocabulary_summary.csv", index=False)

top_words = pd.DataFrame(word_counts.most_common(TOP_N_WORDS), columns=["token","frequency"])
display(top_words)
top_words.to_csv(TABLES_DIR/"top_words.csv", index=False)

plt.figure(figsize=(11, 8))
sns.barplot(data=top_words, x="frequency", y="token")
plt.title("Most frequent normalized Arabic tokens")
plt.tight_layout()
plt.savefig(PLOTS_DIR/"top_words.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()

def ngram_counts(token_lists, n):
    result = Counter()
    for tokens in token_lists:
        result.update(tuple(tokens[i:i+n]) for i in range(max(0, len(tokens)-n+1)))
    return result

bigrams = pd.DataFrame([(" ".join(k),v) for k,v in ngram_counts(df.tokens,2).most_common(25)],
                       columns=["bigram","frequency"])
trigrams = pd.DataFrame([(" ".join(k),v) for k,v in ngram_counts(df.tokens,3).most_common(25)],
                        columns=["trigram","frequency"])
display(bigrams)
display(trigrams)
bigrams.to_csv(TABLES_DIR/"top_bigrams.csv", index=False)
trigrams.to_csv(TABLES_DIR/"top_trigrams.csv", index=False)

## 10. Character inventory and script profile

In [ ]:
all_text = " ".join(df.transcript_original)
char_counts = Counter(all_text)
char_table = pd.DataFrame([
    ("<SPACE>" if ch.isspace() else ch, f"U+{ord(ch):04X}", count)
    for ch,count in char_counts.most_common(TOP_N_CHARACTERS)
], columns=["character","unicode","frequency"])
display(char_table)
char_table.to_csv(TABLES_DIR/"character_inventory.csv", index=False)

punctuation = list(".,!?؟،؛:()[]{}«»\"'ـ…")
punct_table = pd.DataFrame([(ch,all_text.count(ch)) for ch in punctuation if all_text.count(ch)],
                           columns=["punctuation","frequency"]).sort_values("frequency",ascending=False)
display(punct_table)
punct_table.to_csv(TABLES_DIR/"punctuation_frequency.csv", index=False)

script_profile = pd.DataFrame([
    ("Arabic block characters", len(re.findall(r"[\u0600-\u06FF]", all_text))),
    ("Latin letters", len(re.findall(r"[A-Za-z]", all_text))),
    ("Digits", len(re.findall(r"\d", all_text))),
    ("Whitespace", sum(ch.isspace() for ch in all_text))
], columns=["group","count"])
display(script_profile)

## 11. Contributor and demographic metadata
`client_id` is a dataset-level contributor key. Gender, age, and accent are analyzed only when supplied; missing values remain visible. These values are not independently verified.

In [ ]:
if SPEAKER_COL:
    contributor_summary = df.groupby(SPEAKER_COL, dropna=False).agg(
        clip_count=("transcript_original","size"),
        mean_words=("word_count","mean"),
        median_words=("word_count","median")
    ).reset_index().sort_values("clip_count",ascending=False)
    print("Unique contributor IDs:", df[SPEAKER_COL].nunique(dropna=True))
    display(contributor_summary.head(25))
    contributor_summary.to_csv(TABLES_DIR/"contributor_summary.csv", index=False)

    fig, axes = plt.subplots(1,2,figsize=(14,4))
    sns.histplot(contributor_summary.clip_count,bins=30,ax=axes[0])
    axes[0].set_title("Clips per contributor")
    sns.barplot(data=contributor_summary.head(20),x="clip_count",y=SPEAKER_COL,ax=axes[1])
    axes[1].set_title("Top 20 contributors")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR/"contributor_distribution.png",dpi=FIGURE_DPI,bbox_inches="tight")
    plt.show()
else:
    print("No contributor ID field detected.")

for col, label in [(GENDER_COL,"gender"),(AGE_COL,"age"),(ACCENT_COL,"accent"),(LOCALE_COL,"locale")]:
    if col:
        counts = df[col].fillna("Missing").astype(str).value_counts().rename_axis(label).reset_index(name="clips")
        display(counts.head(30))
        counts.to_csv(TABLES_DIR/f"{label}_distribution.csv",index=False)
        if len(counts) <= 25:
            plt.figure(figsize=(10,max(4,len(counts)*.28)))
            sns.barplot(data=counts,x="clips",y=label)
            plt.title(f"{label.title()} metadata distribution")
            plt.tight_layout()
            plt.savefig(PLOTS_DIR/f"{label}_distribution.png",dpi=FIGURE_DPI,bbox_inches="tight")
            plt.show()
    else:
        print(label, "metadata not available.")

## 12. Community votes
Up-votes and down-votes are contribution metadata. They may reflect community feedback, but they are not direct measurements of acoustic quality or transcript correctness.

In [ ]:
if UPVOTES_COL or DOWNVOTES_COL:
    vote_columns = [c for c in [UPVOTES_COL, DOWNVOTES_COL] if c]
    for col in vote_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    display(df[vote_columns].describe().T)
    if UPVOTES_COL and DOWNVOTES_COL:
        df["total_votes"] = df[UPVOTES_COL].fillna(0)+df[DOWNVOTES_COL].fillna(0)
        df["upvote_ratio"] = np.where(df.total_votes>0,df[UPVOTES_COL].fillna(0)/df.total_votes,np.nan)
        display(df[["total_votes","upvote_ratio"]].describe().T)
        df[["total_votes","upvote_ratio"]].to_csv(TABLES_DIR/"vote_metrics.csv",index=False)
else:
    print("Vote fields unavailable.")

## 13. Transcript and metadata quality audit
Duplicate sentences are not automatically errors: multiple speakers may read the same prompt. Flags are intended for manual inspection.

In [ ]:
quality = pd.DataFrame(index=df.index)
quality["empty_transcript"] = df.transcript_original.str.strip().eq("")
quality["duplicate_nonempty_transcript"] = df.transcript_original.duplicated(keep=False) & df.transcript_original.str.strip().ne("")
quality["very_short_le_2_words"] = df.word_count <= 2
quality["very_long_gt_40_words"] = df.word_count > 40
quality["replacement_character"] = df.transcript_original.str.contains("\uFFFD",regex=False)
quality["contains_latin_letters"] = df.transcript_original.str.contains(r"[A-Za-z]",regex=True)
quality["control_characters"] = df.transcript_original.str.contains(r"[\x00-\x08\x0B\x0C\x0E-\x1F]",regex=True)
for col in [SPEAKER_COL,GENDER_COL,AGE_COL,ACCENT_COL,LOCALE_COL,UPVOTES_COL,DOWNVOTES_COL]:
    if col:
        quality[f"missing_{col}"] = df[col].isna()

quality_summary = pd.DataFrame({
    "issue":quality.columns,
    "flagged_records":[int(quality[c].sum()) for c in quality],
    "percent":[float(quality[c].mean()*100) for c in quality]
})
display(quality_summary)
quality_summary.to_csv(TABLES_DIR/"transcript_metadata_quality_audit.csv",index=False)
pd.concat([df[[c for c in [SPEAKER_COL,GENDER_COL,AGE_COL,ACCENT_COL,TRANSCRIPT_COL] if c]].copy(),quality],axis=1).to_csv(
    TABLES_DIR/"record_level_quality_flags.csv",index=False
)

## 14. Select representative audio sample
Random sampling is reproducible. Stratified sampling proportionally samples across `client_id` when available.

In [ ]:
def choose_indices(frame, n, strategy="random", group_col=None, seed=42):
    n = min(int(n),len(frame))
    if n == len(frame):
        return np.arange(len(frame))
    rng = np.random.default_rng(seed)
    if strategy == "stratified" and group_col:
        groups = frame.groupby(group_col,dropna=False).indices
        exact = {k:len(v)*n/len(frame) for k,v in groups.items()}
        alloc = {k:int(np.floor(v)) for k,v in exact.items()}
        remainder = n-sum(alloc.values())
        for k in sorted(exact,key=lambda k:exact[k]-alloc[k],reverse=True)[:remainder]:
            alloc[k]+=1
        selected=[]
        for k,idx in groups.items():
            take=min(alloc[k],len(idx))
            if take:
                selected.extend(rng.choice(np.asarray(idx),size=take,replace=False).tolist())
        return np.asarray(selected[:n],dtype=int)
    if strategy == "stratified":
        print("No group column found; using random sampling.")
    return rng.choice(len(frame),size=n,replace=False)

group_col = STRATIFY_COLUMN if STRATIFY_COLUMN in df.columns else SPEAKER_COL
sample_indices = np.sort(choose_indices(df,AUDIO_SAMPLE_SIZE,AUDIO_SAMPLING_STRATEGY,group_col,RANDOM_SEED))
print("Selected audio rows:",len(sample_indices),"| group field:",group_col)

## 15. Decode sampled audio and inspect signal metadata
Only sampled audio is decoded. The cell supports decoded arrays and file paths.

In [ ]:
if AUDIO_COL is None:
    raise ValueError("No audio column found.")

def decode_payload(payload):
    if isinstance(payload,dict):
        if payload.get("array") is not None:
            return np.asarray(payload["array"]),int(payload["sampling_rate"])
        if payload.get("path"):
            x,sr=sf.read(payload["path"],always_2d=False)
            return np.asarray(x),int(sr)
    if isinstance(payload,str):
        x,sr=sf.read(payload,always_2d=False)
        return np.asarray(x),int(sr)
    raise TypeError(f"Unsupported audio payload type: {type(payload)}")

audio_arrays={}
audio_rows=[]
for j,idx in enumerate(sample_indices,1):
    try:
        x,sr=decode_payload(train_ds[int(idx)][AUDIO_COL])
        if x.ndim==2:
            channels=x.shape[1] if x.shape[0]>x.shape[1] else x.shape[0]
            mono=x.mean(axis=1) if x.shape[0]>x.shape[1] else x.mean(axis=0)
        else:
            channels=1
            mono=x
        mono=np.asarray(mono,dtype=np.float32)
        audio_arrays[int(idx)]=(mono,sr)
        audio_rows.append({
            "dataset_index":int(idx),"sample_rate":sr,"channels":channels,
            "num_samples":len(mono),"duration_seconds":len(mono)/sr,
            "peak_amplitude":float(np.max(np.abs(mono))) if len(mono) else np.nan,
            "rms_amplitude":float(np.sqrt(np.mean(mono**2))) if len(mono) else np.nan,
            "decode_error":None
        })
    except Exception as exc:
        audio_rows.append({"dataset_index":int(idx),"sample_rate":np.nan,"channels":np.nan,
                           "num_samples":np.nan,"duration_seconds":np.nan,
                           "peak_amplitude":np.nan,"rms_amplitude":np.nan,"decode_error":str(exc)})
    if j%50==0 or j==len(sample_indices):
        print(f"{j}/{len(sample_indices)}")
audio_df=pd.DataFrame(audio_rows)
display(audio_df.head())
print("Decoded:",len(audio_arrays),"Failures:",audio_df.duration_seconds.isna().sum())
audio_df.to_csv(TABLES_DIR/"audio_sample_metadata.csv",index=False)

## 16. Audio duration, sample rate, channels, and amplitude

In [ ]:
display(audio_df[["duration_seconds","sample_rate","channels","peak_amplitude","rms_amplitude"]].describe().T)
fig,axes=plt.subplots(2,2,figsize=(14,8))
sns.histplot(audio_df.duration_seconds.dropna(),bins=30,kde=True,ax=axes[0,0]); axes[0,0].set_title("Duration (seconds)")
sns.countplot(data=audio_df.dropna(subset=["sample_rate"]),x="sample_rate",ax=axes[0,1]); axes[0,1].set_title("Sample rate"); axes[0,1].tick_params(axis="x",rotation=30)
sns.countplot(data=audio_df.dropna(subset=["channels"]),x="channels",ax=axes[1,0]); axes[1,0].set_title("Channels")
sns.histplot(audio_df.rms_amplitude.dropna(),bins=30,kde=True,ax=axes[1,1]); axes[1,1].set_title("RMS amplitude")
plt.tight_layout()
plt.savefig(PLOTS_DIR/"audio_distributions.png",dpi=FIGURE_DPI,bbox_inches="tight")
plt.show()

## 17. Listen to audio and visualize waveforms/spectrograms

In [ ]:
for idx in list(audio_arrays)[:N_AUDIO_EXAMPLES]:
    signal,sr=audio_arrays[idx]
    duration=len(signal)/sr
    display(Markdown(f"### Sample row {idx} · {duration:.2f} seconds · {sr} Hz"))
    display(Audio(signal,rate=sr))
    crop=signal[:int(min(MAX_WAVEFORM_SECONDS,duration)*sr)]
    fig,axes=plt.subplots(2,1,figsize=(13,6))
    axes[0].plot(np.arange(len(crop))/sr,crop,linewidth=.6)
    axes[0].set_title("Waveform"); axes[0].set_xlabel("Time (s)")
    if len(crop):
        spec=librosa.stft(crop)
        db=librosa.amplitude_to_db(np.abs(spec),ref=np.max)
        img=librosa.display.specshow(db,sr=sr,x_axis="time",y_axis="hz",ax=axes[1])
        axes[1].set_title("Magnitude spectrogram")
        fig.colorbar(img,ax=axes[1],format="%+2.0f dB")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR/f"waveform_spectrogram_{idx}.png",dpi=FIGURE_DPI,bbox_inches="tight")
    plt.show()

## 18. Low-energy region inspection
A relative frame-energy threshold is a rough diagnostic, not a voice-activity detector. Quiet speech may also be flagged.

In [ ]:
def low_energy_fraction(x,frame_length=2048,hop_length=512,threshold_db=-40):
    if len(x)<frame_length: return np.nan
    rms=librosa.feature.rms(y=x,frame_length=frame_length,hop_length=hop_length)[0]
    if not len(rms) or np.max(rms)==0: return 1.0
    threshold=np.max(rms)*10**(threshold_db/20)
    return float(np.mean(rms<threshold))

low_df=pd.DataFrame([{"dataset_index":i,"low_energy_fraction":low_energy_fraction(x)}
                     for i,(x,sr) in audio_arrays.items()])
audio_df=audio_df.merge(low_df,on="dataset_index",how="left")
display(audio_df[["duration_seconds","rms_amplitude","low_energy_fraction"]].describe().T)
sns.histplot(audio_df.low_energy_fraction.dropna(),bins=25,kde=True)
plt.title("Estimated low-energy frame fraction")
plt.savefig(PLOTS_DIR/"low_energy_fraction.png",dpi=FIGURE_DPI,bbox_inches="tight")
plt.show()

## 19. Text-audio relationship and estimated speaking rate

In [ ]:
audio_text=audio_df.merge(df.reset_index().rename(columns={"index":"dataset_index"}),
                          on="dataset_index",how="left",suffixes=("_audio",""))
audio_text["words_per_second"]=audio_text.word_count/audio_text.duration_seconds.replace(0,np.nan)
audio_text["characters_per_second"]=audio_text.character_count/audio_text.duration_seconds.replace(0,np.nan)
display(audio_text[["duration_seconds","word_count","character_count","words_per_second","characters_per_second"]].describe().T)

for xcol,title,filename in [
    ("word_count","Word count vs duration","word_count_vs_duration.png"),
    ("character_count","Character count vs duration","character_count_vs_duration.png")
]:
    plt.figure(figsize=(8,5))
    sns.regplot(data=audio_text,x=xcol,y="duration_seconds",scatter_kws={"alpha":.4},line_kws={"color":"red"})
    plt.title(title); plt.tight_layout()
    plt.savefig(PLOTS_DIR/filename,dpi=FIGURE_DPI,bbox_inches="tight")
    plt.show()

corr=[]
for col in ["word_count","character_count","normalized_character_count"]:
    valid=audio_text[[col,"duration_seconds"]].dropna()
    if len(valid)>=3:
        p=pearsonr(valid[col],valid.duration_seconds)
        s=spearmanr(valid[col],valid.duration_seconds)
        corr.append({"measure":col,"n":len(valid),"pearson_r":p.statistic,"pearson_p":p.pvalue,
                     "spearman_rho":s.statistic,"spearman_p":s.pvalue})
correlation_table=pd.DataFrame(corr)
display(correlation_table)
correlation_table.to_csv(TABLES_DIR/"text_audio_correlations.csv",index=False)

sns.histplot(audio_text.words_per_second.replace([np.inf,-np.inf],np.nan).dropna(),bins=30,kde=True)
plt.title("Estimated words per second")
plt.savefig(PLOTS_DIR/"speaking_rate.png",dpi=FIGURE_DPI,bbox_inches="tight")
plt.show()

## 20. Audio quality audit
Thresholds are heuristics for inspection, not automatic exclusion criteria.

In [ ]:
audio_quality=audio_df.copy()
audio_quality["decode_failed"]=audio_quality.duration_seconds.isna()
audio_quality["very_short_audio"]=audio_quality.duration_seconds<1.0
audio_quality["long_audio"]=audio_quality.duration_seconds>20.0
audio_quality["near_zero_rms"]=audio_quality.rms_amplitude<1e-4
audio_quality["unexpected_sample_rate"]=~audio_quality.sample_rate.isin([8000,16000,22050,24000,32000,44100,48000])
flags=["decode_failed","very_short_audio","long_audio","near_zero_rms","unexpected_sample_rate"]
audio_quality_summary=pd.DataFrame({"issue":flags,"flagged_clips":[int(audio_quality[c].fillna(False).sum()) for c in flags]})
display(audio_quality_summary)
audio_quality_summary.to_csv(TABLES_DIR/"audio_quality_audit_sample.csv",index=False)

## 21. Contributor-level audio comparisons

In [ ]:
if SPEAKER_COL:
    contributor_audio=audio_text.groupby(SPEAKER_COL).agg(
        sampled_clips=("duration_seconds","count"),
        mean_duration=("duration_seconds","mean"),
        mean_words_per_second=("words_per_second","mean")
    ).reset_index().sort_values("sampled_clips",ascending=False)
    display(contributor_audio.head(25))
    contributor_audio.to_csv(TABLES_DIR/"contributor_audio_summary_sample.csv",index=False)
else:
    print("No contributor identifier available.")

## 22. Optional train/validation/test comparison

In [ ]:
split_frames={"train":df}
for split_name, split_ds in additional_splits.items():
    meta=split_ds.remove_columns("audio") if "audio" in split_ds.column_names else split_ds
    tmp=meta.to_pandas()
    tcol=first_existing(["sentence","transcription","raw_transcription","text"],tmp.columns)
    if tcol:
        tmp["word_count"]=tmp[tcol].fillna("").astype(str).map(lambda x:len(normalize_arabic_light(x).split()))
    split_frames[split_name]=tmp

split_comparison=pd.DataFrame([
    {"split":name,"rows":len(frame),
     "mean_words":frame.word_count.mean() if "word_count" in frame else np.nan,
     "median_words":frame.word_count.median() if "word_count" in frame else np.nan}
    for name,frame in split_frames.items()
])
display(split_comparison)
split_comparison.to_csv(TABLES_DIR/"split_comparison.csv",index=False)

## 23. Generate a data-driven Markdown summary

In [ ]:
summary_lines=[
    "# Mozilla Common Voice 15.0 Arabic: EDA summary","",
    "## Dataset",
    f"- Repository: `{DATASET_ID}`",
    f"- Configuration: `{CONFIG_NAME}`",
    f"- Primary split: `{PRIMARY_SPLIT}`",
    f"- Training records: **{len(df):,}**",
    f"- Metadata columns: **{len(df.columns)}**","",
    "## Transcripts",
    f"- Mean words per sentence: **{df.word_count.mean():.2f}**",
    f"- Median words per sentence: **{df.word_count.median():.2f}**",
    f"- Vocabulary size: **{len(word_counts):,}**",
    f"- Total whitespace tokens: **{len(all_tokens):,}**","",
    "## Contributor metadata",
    f"- Contributor ID field: **{SPEAKER_COL or 'Unavailable'}**",
    f"- Unique contributor IDs: **{df[SPEAKER_COL].nunique(dropna=True):,}**" if SPEAKER_COL else "- Unique contributor IDs: unavailable",
    f"- Gender: **{GENDER_COL or 'Unavailable'}** | Age: **{AGE_COL or 'Unavailable'}** | Accent: **{ACCENT_COL or 'Unavailable'}**","",
    "## Audio sample",
    f"- Requested sample: **{AUDIO_SAMPLE_SIZE}**",
    f"- Successfully decoded: **{len(audio_arrays)}**",
    f"- Median duration: **{audio_df.duration_seconds.median():.2f} seconds**",
    f"- Mean duration: **{audio_df.duration_seconds.mean():.2f} seconds**","",
    "## Quality flags",
    f"- Empty transcripts: **{int(df.transcript_original.str.strip().eq('').sum()):,}**",
    f"- Duplicate non-empty transcript records: **{int((df.transcript_original.duplicated(keep=False)&df.transcript_original.str.strip().ne('')).sum()):,}**",
    f"- Audio decode failures in sample: **{int(audio_df.duration_seconds.isna().sum()):,}**","",
    "## Interpretation cautions",
    "- Full training split is used for transcript/metadata analysis; audio analysis is sample-based.",
    "- Duplicate sentences can be legitimate because several contributors may read the same prompt.",
    "- Contributor IDs are dataset keys, not real-world identities.",
    "- Votes are not direct acoustic-quality or transcription-accuracy scores.",
    "- Whitespace tokenization is not Arabic morphological segmentation.",
    "- Words per second is an approximate text-duration ratio, not a phonetic speaking-rate measure."
]
summary_text="\n".join(summary_lines)
display(Markdown(summary_text))
(REPORT_DIR/"eda_summary.md").write_text(summary_text,encoding="utf-8")

## 24. Export tables and run manifest

In [ ]:
export_cols=[c for c in [SPEAKER_COL,GENDER_COL,AGE_COL,ACCENT_COL,LOCALE_COL,UPVOTES_COL,DOWNVOTES_COL,
                          "transcript_original","transcript_normalized","word_count","character_count","unique_word_count"]
             if c and c in df.columns]
df[export_cols].to_csv(TABLES_DIR/"all_train_transcript_metadata.csv",index=False)

manifest={
    "dataset_id":DATASET_ID,"config":CONFIG_NAME,"primary_split":PRIMARY_SPLIT,
    "train_rows":int(len(df)),"audio_sample_requested":int(AUDIO_SAMPLE_SIZE),
    "audio_sample_decoded":int(len(audio_arrays)),"sampling_strategy":AUDIO_SAMPLING_STRATEGY,
    "stratification_column":group_col,"random_seed":int(RANDOM_SEED),
    "detected_fields":detected,"additional_splits_loaded":list(additional_splits)
}
(OUTPUT_DIR/"run_manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
print("Artifacts saved under",OUTPUT_DIR)

## 25. Generate a standalone PDF report
This report contains the computed summary, selected charts, and selected tables. It is distinct from the full notebook PDF.

In [ ]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, PageBreak
from xml.sax.saxutils import escape

styles=getSampleStyleSheet()
styles.add(ParagraphStyle(name="SmallCell",parent=styles["BodyText"],fontSize=7,leading=9))
story=[Paragraph("Mozilla Common Voice 15.0 Arabic",styles["Title"]),
       Paragraph("Exploratory Data Analysis | Training split",styles["Heading2"]),Spacer(1,.3*cm)]
for line in summary_lines:
    if line.startswith("# "): continue
    if line.startswith("## "): story.append(Paragraph(escape(line[3:]),styles["Heading2"]))
    elif line.startswith("- "): story.append(Paragraph("• "+escape(line[2:]),styles["BodyText"]))
    elif line.strip(): story.append(Paragraph(escape(line),styles["BodyText"]))
    else: story.append(Spacer(1,.1*cm))

story.append(PageBreak())
story.append(Paragraph("Selected visualizations",styles["Heading1"]))
for filename in ["missing_values.png","transcript_length_distributions.png","top_words.png",
                 "contributor_distribution.png","gender_distribution.png","age_distribution.png",
                 "accent_distribution.png","audio_distributions.png","low_energy_fraction.png",
                 "word_count_vs_duration.png","character_count_vs_duration.png","speaking_rate.png"]:
    path=PLOTS_DIR/filename
    if path.exists():
        story.append(Paragraph(escape(filename.replace("_"," ").replace(".png","").title()),styles["Heading3"]))
        image=RLImage(str(path),width=16.5*cm,height=8*cm)
        image.hAlign="CENTER"
        story.extend([image,Spacer(1,.2*cm)])

def add_table(csv_path,title,max_rows=18):
    path=Path(csv_path)
    if not path.exists(): return
    data=pd.read_csv(path).head(max_rows)
    if data.empty: return
    story.append(PageBreak())
    story.append(Paragraph(escape(title),styles["Heading2"]))
    rows=[list(data.columns)]+data.astype(str).values.tolist()
    wrapped=[[Paragraph(escape(str(v)),styles["SmallCell"]) for v in row] for row in rows]
    table=Table(wrapped,repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND",(0,0),(-1,0),colors.HexColor("#243B53")),
        ("TEXTCOLOR",(0,0),(-1,0),colors.white),
        ("GRID",(0,0),(-1,-1),.3,colors.grey),
        ("VALIGN",(0,0),(-1,-1),"TOP"),
        ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.white,colors.HexColor("#F2F5F8")])
    ]))
    story.append(table)

add_table(TABLES_DIR/"column_overview.csv","Dataset schema and missingness")
add_table(TABLES_DIR/"transcript_length_summary.csv","Transcript length statistics")
add_table(TABLES_DIR/"top_words.csv","Most frequent tokens")
add_table(TABLES_DIR/"text_audio_correlations.csv","Text-audio correlations")
add_table(TABLES_DIR/"transcript_metadata_quality_audit.csv","Transcript and metadata quality audit")
add_table(TABLES_DIR/"audio_quality_audit_sample.csv","Audio quality audit")

pdf_path=REPORT_DIR/"Common_Voice_15_Arabic_EDA_Report.pdf"
SimpleDocTemplate(str(pdf_path),pagesize=A4,rightMargin=1.5*cm,leftMargin=1.5*cm,
                  topMargin=1.5*cm,bottomMargin=1.5*cm).build(story)
print("PDF generated:",pdf_path)

## 26. Export the executed notebook itself to PDF
The standalone PDF above is a reader-friendly report. To export the complete executed notebook, including code and outputs:

**In Colab:** run all cells, save the notebook, choose **File → Print** (or browser print), then select **Save as PDF**.

**Locally:** download the `.ipynb` and run:
```bash
python -m pip install -U nbconvert
jupyter nbconvert --to webpdf --allow-chromium-download Common_Voice_15_Arabic_EDA.ipynb
```
If webpdf is unavailable, use `jupyter nbconvert --to html ...` and print the HTML from a browser.

## 27. Final checklist and interpretation notes
- Confirm repository/configuration: `fsicoli/common_voice_15_0`, `ar`.
- Confirm `train` split and detected transcript field.
- Inspect original/normalized transcript pairs.
- Review availability and missingness of client, gender, age, accent, locale, and vote fields.
- Confirm sample size and sampling strategy.
- Listen to flagged audio and inspect corresponding transcripts.
- Review flags before excluding any records.
- Generate both the standalone PDF and, if required, the full notebook PDF.

Do not infer model performance from EDA. Use a documented normalization protocol and held-out evaluation split for ASR experiments. Audio results are sample-based unless the sample is expanded to the entire split.